# Araba Fiyatları (Car Prices)

🎯 Bu challenge’ın amacı, bir dataset hazırlamak ve şimdiye kadar öğrendiğiniz bazı feature selection tekniklerini uygulamaktır.

🚗 Arabalarla ilgili bir veri setiyle çalışıyoruz ve bir arabanın pahalı mı yoksa ucuz mu olduğunu tahmin etmek istiyoruz.

In [1]:
# Data manipulation
import numpy as np
import pandas as pd
# Data visualisation
import matplotlib.pyplot as plt
import seaborn as sns
# Sayısal bir özelliğin normal dağılım gösterip göstermediğini kontrol etme
from statsmodels.graphics.gofplots import qqplot


In [2]:
url = "https://d32aokrjazspmn.cloudfront.net/materials/ML_Cars_dataset.csv"

❓ CSV dosyasını `df` adlı bir veri çerçevesine yükleyin.

In [3]:
import pandas as pd
import urllib.request
import ssl

# SSL doğrulamasını devre dışı bırakıyoruz
ssl_context = ssl._create_unverified_context()

url = "https://d32aokrjazspmn.cloudfront.net/materials/ML_Cars_dataset.csv"

# urlopen ile veriyi çekip, doğrudan read_csv içine okutuyoruz
response = urllib.request.urlopen(url, context=ssl_context)
df = pd.read_csv(response)

# İlk 5 satırı görelim
df.head()

,aspiration,enginelocation,carwidth,curbweight,enginetype,cylindernumber,stroke,peakrpm,price
0,std,front,64.1,2548,dohc,four,2.68,5000,expensive
1,std,front,64.1,2548,dohc,four,2.68,5000,expensive
2,std,front,65.5,2823,ohcv,six,3.47,5000,expensive
3,std,front,NaN,2337,ohc,four,3.40,5500,expensive
4,std,front,66.4,2824,ohc,five,3.40,5500,expensive


ℹ️ Dataset’in açıklaması [burada](https://drive.google.com/file/d/1ADSyjWfRGYqdXwCCN4PPC7PjQeMZ-ap-/view?usp=sharing ) mevcuttur. Egzersiz boyunca buna mutlaka referans verin.

## (1) Yinelenenler (Duplicates)

❓ Varsa, veri kümesinden yinelenenleri kaldırın. ❓

*Veri çerçevesini `df`* üzerine yazın.

In [4]:
# Yinelenen satırları kaldırıp df üzerine yazıyoruz
df = df.drop_duplicates()

# Ne kadar satır/sütun kaldığını görmek için (opsiyonel)
print("Güncel veri boyutu:", df.shape)

Güncel veri boyutu: (191, 9)


## (2)  Eksik değerler (Missing values)

❓ Eksik değerleri bulun ve bunları ya `strategy = "most frequent"` (kategorik değişkenler için) ya da `strategy = "median"` (sayısal değişkenler için) kullanarak doldurun ❓

In [5]:
# Her bir sütundaki eksik değer sayısını bulma
df.isnull().sum()

aspiration         0
enginelocation    10
carwidth           2
curbweight         0
enginetype         0
cylindernumber     0
stroke             0
peakrpm            0
price              0
dtype: int64

### `carwidth`

<details>
    <summary> 💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>carwidth</code> sütununda eksik değerler birden fazla şekilde temsil edilmektedir. Bazıları <code>np.nan</code>, bazıları ise <code>*</code> olarak yer alır. Bunlar tespit edildikten sonra, eksik değerler verinin %30’undan daha azını oluşturduğu için medyan değerle doldurulabilir.
</details>

In [6]:
import numpy as np

# 1. '*' karakterlerini gerçek eksik değere (NaN) çeviriyoruz
df['carwidth'] = df['carwidth'].replace('*', np.nan)

# 2. Sütunu sayısal (float) formata dönüştürüyoruz 
# (İçinde * olduğu için veri tipi metin (object) olarak kalmış olabilir)
df['carwidth'] = df['carwidth'].astype(float)

# 3. Medyan değerini hesaplayıp NaN olan yerleri dolduruyoruz (strategy = "median")
df['carwidth'] = df['carwidth'].fillna(df['carwidth'].median())

# Kontrol edelim
print("carwidth sütununda kalan eksik değer:", df['carwidth'].isnull().sum())

carwidth sütununda kalan eksik değer: 0


### `enginelocation`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>enginelocation</code> kategorik bir feature olduğundan ve kategorilerin büyük çoğunluğu <code>front</code> olduğu için, en sık görülen değerle doldurun.
</details>

In [7]:
# 1. Sütundaki en sık tekrar eden (most frequent) değeri buluyoruz
most_frequent_location = df['enginelocation'].mode()[0]

# 2. Eksik yerleri bu değerle dolduruyoruz
df['enginelocation'] = df['enginelocation'].fillna(most_frequent_location)

# Kontrol edelim
print("enginelocation sütununda kalan eksik değer:", df['enginelocation'].isnull().sum())

enginelocation sütununda kalan eksik değer: 0


🧪 **Kodunu test et**

In [8]:
from nbresult import ChallengeResult

result = ChallengeResult('missing_values',
                         dataset = df)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/macos/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/macos/Desktop/S16D2-S-Data-car-prices/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 2 items

test_missing_values.py::TestMissing_values::test_carwidth PASSED         [ 50%]
test_missing_values.py::TestMissing_values::test_engine_location PASSED  [100%]

============================== 2 passed in 0.92s ===============================


💯 You can commit your code:

git add tests/missing_values.pickle

git commit -m 'Completed missing_values step'

git push origin master



## (3) Sayısal özelliklerin ölçeklendirilmesi (Scaling the numerical features)

In [9]:
# Hatırlatma olarak, DataFrame hakkında bazı bilgiler
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 191 entries, 0 to 204
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   aspiration      191 non-null    object 
 1   enginelocation  191 non-null    object 
 2   carwidth        191 non-null    float64
 3   curbweight      191 non-null    int64  
 4   enginetype      191 non-null    object 
 5   cylindernumber  191 non-null    object 
 6   stroke          191 non-null    float64
 7   peakrpm         191 non-null    int64  
 8   price           191 non-null    object 
dtypes: float64(2), int64(2), object(5)
memory usage: 14.9+ KB


In [10]:
# Ve işte ölçeklendirmemiz gereken veri kümesinin sayısal özellikleri
numerical_features = df.select_dtypes(exclude=['object']).columns
numerical_features

Index(['carwidth', 'curbweight', 'stroke', 'peakrpm'], dtype='object')

❓ **Soru: Sayısal feature’ların ölçeklenmesi** ❓

Sayısal feature’ları aykırı değerler (outliers) ve dağılımları açısından inceleyin ve duruma göre aşağıdaki yöntemleri uygulayın:
- Robust Scaler
- Standard Scaler

Dönüştürülmüş değerlerle orijinal sütunları değiştirin.

### `peakrpm` , `carwidth` , & `stroke`

<details>
    <summary>💡 <i>İpucu</i> </summary>

    
ℹ️ <code>peakrpm</code>, <code>carwidth</code> ve <code>stroke</code> normal dağılıma sahiptir ancak aynı zamanda bazı aykırı değerler (outlier) içerir. Bu nedenle `RobustScaler()` kullanılması tavsiye edilir.
</details>

In [11]:
from sklearn.preprocessing import RobustScaler

robust_scaler = RobustScaler()
cols_to_robust = ['peakrpm', 'carwidth', 'stroke']
df[cols_to_robust] = robust_scaler.fit_transform(df[cols_to_robust])

In [12]:
# Yaptığımız değişikliği görelim
df[cols_to_robust].head()

,peakrpm,carwidth,stroke
0,-0.142857,-0.518519,-2.033333
2,-0.142857,0.000000,0.600000
3,0.571429,0.000000,0.366667
4,0.571429,0.333333,0.366667
5,0.571429,0.296296,0.366667


### `curbweight`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>curbweight</code> normal bir dağılıma sahiptir ve aykırı değer (outlier) içermez. Bu nedenle Standard Scaler ile ölçeklenebilir.
</details>

In [13]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
df[['curbweight']] = std_scaler.fit_transform(df[['curbweight']])

In [14]:
# Yaptığımız değişikliği görelim
df[['curbweight']].head()

,curbweight
0,-0.048068
2,0.476395
3,-0.450474
4,0.478302
5,-0.126260


🧪 **Kodunu test et**

In [15]:
from nbresult import ChallengeResult

result = ChallengeResult('scaling',
                         dataset = df
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/macos/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/macos/Desktop/S16D2-S-Data-car-prices/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 4 items

test_scaling.py::TestScaling::test_carwidth PASSED                       [ 25%]
test_scaling.py::TestScaling::test_curbweight PASSED                     [ 50%]
test_scaling.py::TestScaling::test_peakrpm PASSED                        [ 75%]
test_scaling.py::TestScaling::test_stroke PASSED                         [100%]

============================== 4 passed in 0.83s ===============================


💯 You can commit your code:

git add tests/scaling.pickle

git commit -m 'Completed scaling step'

git push origin master



## (4) Kategorik özelliklerin kodlanması (Encoding the categorical features)

❓ **Soru: Kategorik değişkenlerin encode edilmesi** ❓

👇 Encode edilmesi gereken feature’ları inceleyin ve duruma göre aşağıdaki teknikleri uygulayın:

- One-hot encoding
- Manuel ordinal encoding

DataFrame içinde, orijinal feature’ları encode edilmiş versiyonlarıyla değiştirin.

### `aspiration` & `enginelocation`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>aspiration</code> ve <code>enginelocation</code> ikili (binary) kategorik feature’lardır.
</details>

In [16]:
from sklearn.preprocessing import LabelEncoder

# İkili (binary) kategorik değişkenleri 0 ve 1'e dönüştürüyoruz
le = LabelEncoder()

df['aspiration'] = le.fit_transform(df['aspiration'])
df['enginelocation'] = le.fit_transform(df['enginelocation'])

# Değişiklikleri kontrol edelim
df[['aspiration', 'enginelocation']].head()

,aspiration,enginelocation
0,0,0
2,0,0
3,0,0
4,0,0
5,0,0


### `enginetype`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>enginetype</code> çok kategorili (multicategorical) bir feature’dır ve One-hot encoding uygulanmalıdır.
</details>

In [17]:
# One-Hot Encoding uyguluyoruz
# dtype=int kullanarak True/False yerine 0 ve 1 olarak dönüştürülmesini sağlıyoruz
df = pd.get_dummies(df, columns=['enginetype'], dtype=int)

# Veri setinin yeni boyutunu görelim (yeni sütunlar eklendiği için sütun sayısı artacak)
print("Yeni veri boyutu:", df.shape)
df.head()

Yeni veri boyutu: (191, 15)


,aspiration,enginelocation,carwidth,curbweight,cylindernumber,stroke,peakrpm,price,enginetype_dohc,enginetype_dohcv,enginetype_l,enginetype_ohc,enginetype_ohcf,enginetype_ohcv,enginetype_rotor
0,0,0,-0.518519,-0.048068,four,-2.033333,-0.142857,expensive,1,0,0,0,0,0,0
2,0,0,0.000000,0.476395,six,0.600000,-0.142857,expensive,0,0,0,0,0,1,0
3,0,0,0.000000,-0.450474,four,0.366667,0.571429,expensive,0,0,0,1,0,0,0
4,0,0,0.333333,0.478302,five,0.366667,0.571429,expensive,0,0,0,1,0,0,0
5,0,0,0.296296,-0.126260,five,0.366667,0.571429,expensive,0,0,0,1,0,0,0


In [18]:
df.shape

(191, 15)

### `cylindernumber`

<details>
    <summary>💡 İpucu </summary>

ℹ️ <code>cylindernumber</code> sıralı (ordinal) bir feature’dır ve sayısal değerlere manuel olarak encode edilmelidir.

</details>

In [19]:
# İngilizce metin sayıları, gerçek sayılara dönüştürecek bir sözlük oluşturuyoruz
cylinder_dict = {
    'two': 2,
    'three': 3,
    'four': 4,
    'five': 5,
    'six': 6,
    'eight': 8,
    'twelve': 12
}

# map fonksiyonu ile eşleştirmeyi yapıp sütunu güncelliyoruz
df['cylindernumber'] = df['cylindernumber'].map(cylinder_dict)

# Kontrol edelim
df[['cylindernumber']].head()

,cylindernumber
0,4
2,6
3,4
4,5
5,5


❓ Artık `cylindernumber`’ı 2 ile 12 arasında sayısal bir feature’a dönüştürdüğünüze göre, bunu ölçeklendirmeniz gerekiyor ❓

<br/>

<details>
    <summary>💡 İpucu </summary>

`cylindernumber`’ın mevcut dağılımına bakın ve kendinize şu soruları sorun:
- Ölçekleme, bir feature’ın dağılımını etkiler mi?
- Bu feature’ın dağılımına göre en uygun ölçekleme yöntemi hangisidir?
</details>

In [20]:
from sklearn.preprocessing import RobustScaler

# 1. Metinleri sayılara çeviriyoruz (replace kullanmak daha güvenlidir, dönüştüremediğini NaN yapmaz)
cylinder_dict = {
    'two': 2, 'three': 3, 'four': 4, 'five': 5, 
    'six': 6, 'eight': 8, 'twelve': 12
}
df['cylindernumber'] = df['cylindernumber'].replace(cylinder_dict)

# 2. RobustScaler ile ölçeklendiriyoruz
robust_scaler_cyl = RobustScaler()
df[['cylindernumber']] = robust_scaler_cyl.fit_transform(df[['cylindernumber']])

# Sonucu görelim
df[['cylindernumber']].head()

,cylindernumber
0,0.0
2,2.0
3,0.0
4,1.0
5,1.0


<details>
<summary><i>Ölçekleme ve encoding işlemlerinden sonra DataFrame’inizin nasıl görünmesi gerektiğine dair bir ekran görüntüsü aşağıdadır</i></summary>
    
    
<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/02-Prepare-the-dataset/car_price_after_scaling_and_encoding.png">    

</details>

### `price`

👇 Hedef `price`ı kodlayın.

<details>
    <summary>💡 İpucu </summary>
    <br>
    ℹ️ <code>price</code> target değişkendir ve LabelEncoder ile encode edilmelidir.
</details>

In [24]:
from sklearn.preprocessing import LabelEncoder

# LabelEncoder objemizi oluşturuyoruz
le_price = LabelEncoder()

# price sütununu encode edip üzerine yazıyoruz
df['price'] = le_price.fit_transform(df['price'])

# Sonucu görelim (expensive/cheap gibi değerler 0 ve 1'e dönüşecek)
df[['price']].head()

,price
0,1
2,1
3,1
4,1
5,1


🧪 **Kodunu test et**

In [25]:
from nbresult import ChallengeResult

result = ChallengeResult('encoding',
                         dataset = df)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/macos/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/macos/Desktop/S16D2-S-Data-car-prices/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 4 items

test_encoding.py::TestEncoding::test_aspiration PASSED                   [ 25%]
test_encoding.py::TestEncoding::test_enginelocation PASSED               [ 50%]
test_encoding.py::TestEncoding::test_enginetype PASSED                   [ 75%]
test_encoding.py::TestEncoding::test_price PASSED                        [100%]

============================== 4 passed in 0.89s ===============================


💯 You can commit your code:

git add tests/encoding.pickle

git commit -m 'Completed encoding step'

git push origin master



## (5) Temel Modelleme (Base Modelling)

👏 Veri kümesi ön işleme tabi tutuldu ve artık modele uyarlanmaya hazır. 

❓ **Soru: Bir classification modelini ilk kez değerlendirme** ❓

Ön işlenmiş bu dataset üzerinde bir `LogisticRegression` modeli için cross-validation çalıştırın ve elde edilen skoru `base_model_score` adlı değişkende saklayın.

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# 1. Verimizi özellikler (X) ve hedef değişken (y) olarak ayırıyoruz
X = df.drop(columns=['price'])
y = df['price']

# 2. Logistic Regression modelimizi tanımlıyoruz 
# (Modelin rahat çalışması ve uyarı vermemesi için max_iter=1000 ekledik)
log_model = LogisticRegression(max_iter=1000)

# 3. 5 Katlı Cross-Validation (Çapraz doğrulama) uyguluyoruz
scores = cross_val_score(log_model, X, y, cv=5)

# 4. Çıkan 5 farklı skorun ortalamasını alıp hocanın istediği değişkene atıyoruz
base_model_score = scores.mean()

# Sonucu görelim
print("Temel Model Başarı Skoru:", base_model_score)

Temel Model Başarı Skoru: 0.8430499325236166


🧪 **Kodunu test et**

In [27]:
from nbresult import ChallengeResult

result = ChallengeResult('base_model',
                         score = base_model_score
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/macos/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/macos/Desktop/S16D2-S-Data-car-prices/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_base_model.py::TestBase_model::test_base_model_score PASSED         [100%]

============================== 1 passed in 0.20s ===============================


💯 You can commit your code:

git add tests/base_model.pickle

git commit -m 'Completed base_model step'

git push origin master



## (6) Özellik Seçimi  (Feature Selection (with _Permutation Importance_))

👩🏻‍🏫 Bir feature’ın target’ı tahmin etmede gerçekten önemli olup olmadığını tespit etmenin güçlü bir yolu şudur:

1. Bir model çalıştırın ve skorunu ölçün  
2. Bu feature’ı karıştırın (shuffle edin), modeli tekrar çalıştırın ve skoru tekrar ölçün  
    - Eğer performans **belirgin şekilde düşerse**, bu feature önemlidir ve **çıkarılmamalıdır**
    - Eğer performans **çok fazla düşmezse**, bu feature **elenebilir**

❓ **Sorular** ❓

1. Modele en az bilgi katkısı sağlayan feature’ları tespit etmek için feature permutation uygulayın.
2. Model performansının belirgin şekilde düşmeye başladığını fark edene kadar zayıf feature’ları dataset’ten çıkarın.
3. Elde ettiğiniz yeni güçlü feature set’i ile yeni bir modeli cross-validation ile değerlendirin ve skorunu `strong_model_score` adlı değişkende saklayın.

In [30]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score
import pandas as pd

# 1. Permutation importance hesaplıyoruz
log_model.fit(X, y)
result = permutation_importance(log_model, X, y, n_repeats=10, random_state=42)

# Özellikleri önem derecesine göre büyükten küçüğe sıralıyoruz
importances = pd.Series(result.importances_mean, index=X.columns).sort_values(ascending=False)

# 2. En zayıf özellikleri sırasıyla çıkararak skoru test eden döngü
features_to_keep = list(importances.index)
best_score = cross_val_score(log_model, X[features_to_keep], y, cv=5).mean()

# En zayıftan (listenin sonundan) başlayarak geriye doğru ilerliyoruz
for feature in reversed(importances.index):
    # Bu feature'ı çıkardığımızda ne olacağını görmek için geçici bir liste yapalım
    current_features = features_to_keep.copy()
    current_features.remove(feature)
    
    # Yeni listeyle modeli test edelim
    score = cross_val_score(log_model, X[current_features], y, cv=5).mean()
    
    # Eğer bu zayıf özelliği attığımızda skorumuz artıyorsa veya aynı kalıyorsa, gerçekten çöpe atalım!
    if score >= best_score:
        best_score = score
        features_to_keep.remove(feature)
        print(f"🗑️ '{feature}' elendi. Yeni skor: {best_score:.4f}")

# 3. Sonuçları kaydediyoruz
X_strong = X[features_to_keep]
strong_model_score = best_score

print("\n🚀 Güçlü Model Başarı Skoru:", strong_model_score)
print("👑 Kalan Güçlü Özellikler:", X_strong.columns.tolist())

🗑️ 'enginetype_l' elendi. Yeni skor: 0.8430
🗑️ 'enginetype_dohcv' elendi. Yeni skor: 0.8430
🗑️ 'enginelocation' elendi. Yeni skor: 0.8482
🗑️ 'enginetype_ohcv' elendi. Yeni skor: 0.8482
🗑️ 'enginetype_dohc' elendi. Yeni skor: 0.8482
🗑️ 'aspiration' elendi. Yeni skor: 0.8640
🗑️ 'cylindernumber' elendi. Yeni skor: 0.8745
🗑️ 'enginetype_rotor' elendi. Yeni skor: 0.8745
🗑️ 'enginetype_ohc' elendi. Yeni skor: 0.8745
🗑️ 'stroke' elendi. Yeni skor: 0.8903
🗑️ 'carwidth' elendi. Yeni skor: 0.8954

🚀 Güçlü Model Başarı Skoru: 0.8954116059379217
👑 Kalan Güçlü Özellikler: ['curbweight', 'enginetype_ohcf', 'peakrpm']


🧪 **Kodunu test et**

In [31]:
from nbresult import ChallengeResult

result = ChallengeResult('strong_model',
                         score = strong_model_score
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/macos/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/macos/Desktop/S16D2-S-Data-car-prices/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_strong_model.py::TestStrong_model::test_strong_model_score PASSED   [100%]

============================== 1 passed in 0.19s ===============================


💯 You can commit your code:

git add tests/strong_model.pickle

git commit -m 'Completed strong_model step'

git push origin master



## Bonus -  Verilerinizi sınıflandırma (Stratifying your data) ⚖️

💡 Veriyi training ve testing olarak bölerken, dataset’imizdeki kategorik değişkenlerin oranına dikkat etmemiz gerekir — ister target `y`’nin sınıfları olsun ister `X` içindeki kategorik bir feature olsun.

Aşağıda bir örneğe bakalım 👇

❓ Orijinal `X` ve `y` verinizi sklearn’in `train_test_split` fonksiyonunu kullanarak training ve testing olarak ayırın; karşılaştırılabilir sonuçlar elde etmek için `random_state=1` ve `test_size=0.3` kullanın.

In [32]:
from sklearn.model_selection import train_test_split

# Veriyi ayırıyoruz: test boyutu %30, rastgelelik tohumu 1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

❓ Training dataset’inizde ve testing dataset’inizde `price` sınıfı **1** olan araçların oranını kontrol edin.

> _Ham `df` içinde bu orana baktığınızda, yaklaşık **%50 / %50** olması gerekir._

In [33]:
# Eğitim ve test setlerindeki 1 sınıfı (expensive) oranını hesaplıyoruz
train_ratio = y_train.mean()
test_ratio = y_test.mean()

print(f"Training set'teki 1 sınıfı oranı: {train_ratio:.2f}")
print(f"Testing set'teki 1 sınıfı oranı: {test_ratio:.2f}")

Training set'teki 1 sınıfı oranı: 0.50
Testing set'teki 1 sınıfı oranı: 0.52


☝️ Hâlâ yaklaşık olarak **%50 / %50** civarında olmalı.

***Peki random state’i değiştirirsek ne olur?***

❓ `random_state` değerlerini **1’den 10’a** kadar döngüye alın ve her seferinde training ve testing dataset’lerindeki `price` sınıfı **1** olan araçların oranını hesaplayın. ❓

In [34]:
# random_state değerlerini 1'den 10'a kadar döndürüyoruz
for rs in range(1, 11):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=rs)
    
    tr_ratio = y_tr.mean()
    te_ratio = y_te.mean()
    
    print(f"random_state={rs:2} | Train Oranı: %{tr_ratio*100:.1f} | Test Oranı: %{te_ratio*100:.1f}")

random_state= 1 | Train Oranı: %50.4 | Test Oranı: %51.7
random_state= 2 | Train Oranı: %48.1 | Test Oranı: %56.9
random_state= 3 | Train Oranı: %50.4 | Test Oranı: %51.7
random_state= 4 | Train Oranı: %53.4 | Test Oranı: %44.8
random_state= 5 | Train Oranı: %53.4 | Test Oranı: %44.8
random_state= 6 | Train Oranı: %49.6 | Test Oranı: %53.4
random_state= 7 | Train Oranı: %53.4 | Test Oranı: %44.8
random_state= 8 | Train Oranı: %48.9 | Test Oranı: %55.2
random_state= 9 | Train Oranı: %57.9 | Test Oranı: %34.5
random_state=10 | Train Oranı: %48.9 | Test Oranı: %55.2


Her seferinde oranların değiştiğini, hatta bazen oldukça ciddi şekilde değiştiğini gözlemleyeceksiniz 😱! Bu durum model performansını etkileyebilir.

❓ `train_test_split(random_state=1)` kullanılarak eğitilen bir Logistic Regression modelinin test skorunu,  
`random_state=9` kullanılarak eğitilen modelin test skoru ile karşılaştırın ❓

Eğitimi training data üzerinde yapmayı ve skoru testing data üzerinde hesaplamayı unutmayın.

In [35]:
from sklearn.linear_model import LogisticRegression

# --- random_state = 1 için Model ---
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X, y, test_size=0.3, random_state=1)
model_1 = LogisticRegression(max_iter=1000)
model_1.fit(X_train_1, y_train_1)
score_1 = model_1.score(X_test_1, y_test_1)

# --- random_state = 9 için Model ---
X_train_9, X_test_9, y_train_9, y_test_9 = train_test_split(X, y, test_size=0.3, random_state=9)
model_9 = LogisticRegression(max_iter=1000)
model_9.fit(X_train_9, y_train_9)
score_9 = model_9.score(X_test_9, y_test_9)

print(f"random_state=1 ile Test Skoru: {score_1:.4f}")
print(f"random_state=9 ile Test Skoru: {score_9:.4f}")

random_state=1 ile Test Skoru: 0.9310
random_state=9 ile Test Skoru: 0.7931


👀 `random_state=9` ile çok daha düşük bir skor görmelisiniz; çünkü bu test setindeki sınıf **1** araçların oranı %34.5 iken, training setinde bu oran %57.9’a, hatta orijinal dataset’te yaklaşık %50’ye yakındır.

Bu durum oldukça önemlidir; çünkü dataset’te oluşan bu **rastlantısal dengesizlik**, yalnızca model performansını düşürmekle kalmaz, aynı zamanda eğitim veya değerlendirme sırasında “gerçekliği” de bozabilir 🧐

***Peki bu sorunu nasıl çözebiliriz? Tren seti ve test seti arasında sınıfların dağılımını nasıl aynı tutabiliriz? 🔧***

🎁 Neyse ki sklearn’de, estimator (yani model) bir classifier olduğunda ve target bir sınıf olduğunda, bu durum `cross_validate` tarafından otomatik olarak ele alınır. 📚 [**sklearn.model_selection.cross_validate**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html) dokümantasyonunda `cv` parametresini inceleyin.

Çözüm, aşağıdakini kullanmaktır:

> 📚 [**Stratification (Katmanlama)**](https://scikit-learn.org/stable/modules/cross_validation.html#stratification)

### Hedefin tabakalaşması (Stratification of the target)

💡 ***Stratification*** tekniğini `train_test_split` içinde de kullanabiliriz.

❓ Bu kez **1’den 10’a** kadar olan `random_state` döngüsünü tekrar çalıştırın, ancak bu sefer holdout yöntemine ***`stratify=y`*** parametresini de ekleyin. ❓

In [36]:
# random_state değerlerini 1'den 10'a kadar döndürüyoruz (Bu sefer STRATIFY ekliyoruz)
for rs in range(1, 11):
    # stratify=y parametresi ile bölme işlemi
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=rs, stratify=y)
    
    # Oranları tekrar hesaplıyoruz
    tr_ratio = y_tr.mean()
    te_ratio = y_te.mean()
    
    print(f"random_state={rs:2} | Train Oranı: %{tr_ratio*100:.1f} | Test Oranı: %{te_ratio*100:.1f}")

random_state= 1 | Train Oranı: %51.1 | Test Oranı: %50.0
random_state= 2 | Train Oranı: %51.1 | Test Oranı: %50.0
random_state= 3 | Train Oranı: %51.1 | Test Oranı: %50.0
random_state= 4 | Train Oranı: %51.1 | Test Oranı: %50.0
random_state= 5 | Train Oranı: %51.1 | Test Oranı: %50.0
random_state= 6 | Train Oranı: %51.1 | Test Oranı: %50.0
random_state= 7 | Train Oranı: %51.1 | Test Oranı: %50.0
random_state= 8 | Train Oranı: %51.1 | Test Oranı: %50.0
random_state= 9 | Train Oranı: %51.1 | Test Oranı: %50.0
random_state=10 | Train Oranı: %51.1 | Test Oranı: %50.0


👀 Random state değişse bile, training ve testing verilerindeki sınıf oranları, orijinal `y` içindeki oranlarla aynı tutulur. İşte _stratification_ (katmanlama) tam olarak budur.

`train_test_split` fonksiyonunu `stratify` parametresiyle kullandığımızda, training ve testing verileri arasında **bir feature’ın oranlarını da koruyabiliriz**. Bu, özellikle aşağıdaki durumlarda son derece önemlidir:

- Churn tahmininde erkek ve kadın müşteri oranlarını korumak 🙋‍♂️ 🙋
- Ev fiyatlarını tahmin ederken büyük ve küçük evlerin oranlarını korumak 🏠 🏰
- Bir sonraki ürünü önerirken 1–5 arası review score dağılımını (multiclass!) korumak 🛍️
- vb.

Örneğin, bizim dataset’imizde `aspiration` feature’ının training ve testing verilerinde aynı oranda kalmasını istiyorsak, şu şekilde yazabiliriz:

`train_test_split(X, y, test_size=0.3, stratify=X.aspiration)`

---

Gördüğümüz gibi, **`cross_validate` [target değişkeni otomatik olarak stratify edebilir](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html#:~:text=For%20int/None%20inputs%2C%20if%20the%20estimator%20is%20a%20classifier%20and%20y%20is%20either%20binary%20or%20multiclass%2C%20StratifiedKFold%20is%20used.)**, ancak **feature’lar için bunu yapmaz** 🤔 Bunun için biraz ekstra çalışmaya ihtiyacımız var.

Bunun için `StratifiedKFold` kullanmamız gerekiyor 🔬

### Tabakalaşma (Stratification - generalized)

📚 [**StratifiedKFold**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html), veriyi `K` parçaya bölerken belirli sütunlar (feature veya target) üzerinden stratification yapmamıza olanak tanır.

Bu sayede, ilgilendiğimiz kategorik feature’ların oranlarını koruyarak manuel bir cross-validation yapabiliriz — bunu ikili (binary) `aspiration` feature’ı ile deneyelim:

In [37]:
from sklearn.model_selection import StratifiedKFold

# Veriyi 5 fold’a bölecek bir stratified k-fold oluşturma
skf = StratifiedKFold(n_splits=5)
scores = []

# .split() metodu bir iterator oluşturur; 'X.aspiration' stratify edeceğimiz feature’dır
for train_indices, test_indices in skf.split(X, X.aspiration):

    # 'train_indices' ve 'test_indices', orantılı bölünmeler üreten indeks listeleridir
    X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
    y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]

    # modeli başlatma ve eğitme
    model = LogisticRegression()
    model.fit(X_train, y_train)

    # en sonunda 5 fold’un ortalamasını almak için skoru listeye ekleme
    scores.append(model.score(X_test, y_test))

np.array(scores).mean()

0.8585695006747638

📚 [**StratifiedKFold**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html), veriyi `K` parçaya bölerken belirli sütunlar (feature veya target) üzerinden stratification yapmamıza olanak tanır.

Bu sayede, ilgilendiğimiz kategorik feature’ların oranlarını koruyarak manuel bir cross-validation yapabiliriz — bunu ikili (binary) `aspiration` feature’ı ile deneyelim:


🏁 Tebrikler! Tüm veri setini hazırladınız, özellik seçimi yaptınız ve hatta tabakalaşma hakkında bilgi edindiniz 💪.

💾 Not defterinizi git add/commit/push yapmayı unutmayın...

🚀 ... ve bir sonraki challenge'a geçin!